In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

In [ ]:
import pandas as pd
import geopandas as gpd
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

# Load pop_grid

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Build squares from points of grid
grid['geometry'] = grid.apply(lambda row: row.geometry.buffer(50, cap_style='square'), axis=1)
grid['index'] = grid.index

In [ ]:
# Load the OSM supermarkets data
osm_supermarkets = gpd.read_parquet(PATH_PROCESSED / 'supermarkets.parquet')

In [ ]:
# Load gtfs data for public transport stops
stops = pd.read_csv(PATH_RAW / 'gtfs' / 'stops.txt')
stops = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(stops.stop_lon, stops.stop_lat),
    crs='EPSG:4326'
).to_crs('EPSG:3035')

# Include only stops in within Berlin
# stops = stops[stops.intersects(berlin.geometry[0])]


In [ ]:
# Load the rest of the data
routes     = pd.read_csv(PATH_RAW / 'gtfs' / "routes.txt")
trips      = pd.read_csv(PATH_RAW / 'gtfs' / "trips.txt")
stop_times = pd.read_csv(PATH_RAW / 'gtfs' / "stop_times.txt")

In [ ]:
# One representative trip per route+direction (pick first)
rep_trips = (
    trips.groupby(["route_id", "direction_id"], as_index=False)
         .first()[["route_id", "direction_id", "trip_id"]]
)

# Join: route → trip → stop_times → stop coords
seq = (
    rep_trips
    .merge(stop_times[["trip_id", "stop_id", "stop_sequence"]], on="trip_id")
    .merge(stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id")
    .merge(routes[["route_id", "route_short_name", "route_type"]], on="route_id")
    .sort_values(["route_id", "direction_id", "stop_sequence"])
)

In [ ]:
seq.to_csv(PATH_PROCESSED / 'representative_routes.csv', index=False)

In [ ]:
seq

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    stops[stops['parent_station'].isna()].plot(ax=ax, color='royalblue', markersize=1, label='Public Transport Stops', alpha=0.5)
    berlin.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=2, label='Berlin Boundary')
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.2)
    ax.set_axis_off()
    ax.legend()
    plt.title('Public Transport Stops in Berlin', fontsize=16)
    plt.show()


In [ ]:
def gitter_id(row):
    if row['GITTER_ID_100m'] is not None:
        return row['GITTER_ID_100m']
    else: 
        return str(f"CRS3035RES100mN{row['y_mp_100m']}E{row['x_mp_100m']}")
    
grid['cell_id'] = grid.apply(gitter_id, axis=1)

In [ ]:
import partridge as ptg  # pip install partridge

# Bounding box: inner Ring + buffer, in WGS84
BBOX = tuple(berlin.to_crs("EPSG:4326").total_bounds)  # (min_lon, min_lat, max_lon, max_lat)

view = {
    "stops.txt": {
        "stop_lat": lambda x: (x >= BBOX[1]) & (x <= BBOX[3]),
        "stop_lon": lambda x: (x >= BBOX[0]) & (x <= BBOX[2]),
    }
}
feed = ptg.load_feed(str(PATH_RAW / "gtfs"), view=view)
# Then write trimmed feed back out
ptg.writers.write_feed_dangerously(feed, str(PATH_RAW / "gtfs_berlin"))

In [ ]:
import os
import csv
import shutil
import zipfile
from pathlib import Path

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw" / "gtfs").exists():
            return p
    raise FileNotFoundError(
        "Could not find the repo root. Expected data/raw/gtfs."
    )

def gtfs_file_has_data_rows(path: Path) -> bool:
    """
    Return True only if the GTFS CSV has at least one data row beyond the header.
    Header-only files like frequencies.txt will be excluded.
    """
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader, None)
        if header is None:
            return False

        for row in reader:
            if any(cell.strip() for cell in row):
                return True

    return False

# ----- paths -----
REPO_ROOT = find_repo_root()
PATH_RAW = REPO_ROOT / "data" / "raw"
GTFS_DIR = PATH_RAW / "gtfs"
OSM_PBF = PATH_RAW / "berlin-260512.osm.pbf"
GTFS_ZIP = PATH_RAW / "gtfs_berlin.zip"

if not GTFS_DIR.exists():
    raise FileNotFoundError(f"GTFS folder not found: {GTFS_DIR}")
if not OSM_PBF.exists():
    raise FileNotFoundError(f"OSM file not found: {OSM_PBF}")

# ----- clean r5py cache before importing r5py -----
cache_dir = Path.home() / ".cache" / "r5py"
shutil.rmtree(cache_dir, ignore_errors=True)

# JVM options must be set before importing r5py
os.environ["JAVA_TOOL_OPTIONS"] = "-Xmx10G -XX:+UseG1GC"

# ----- rebuild GTFS zip from scratch -----
if GTFS_ZIP.exists():
    GTFS_ZIP.unlink()

included = []
skipped = []

for file in sorted(GTFS_DIR.iterdir()):
    if not file.is_file():
        continue
    if file.suffix.lower() != ".txt":
        continue

    if gtfs_file_has_data_rows(file):
        included.append(file.name)
    else:
        skipped.append(file.name)

if not included:
    raise RuntimeError(
        f"No non-empty GTFS files found in {GTFS_DIR}. "
        "Your feed is probably incomplete."
    )

with zipfile.ZipFile(GTFS_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in included:
        zf.write(GTFS_DIR / name, arcname=name)  # root-level zip entries only

print("GTFS zip created:", GTFS_ZIP)
print("Included:", included)
if skipped:
    print("Skipped empty/header-only files:", skipped)

import r5py

transport_network = r5py.TransportNetwork(
    osm_pbf=OSM_PBF,
    gtfs=[GTFS_ZIP],
)

In [ ]:
'''import os
import sys
import shutil
from pathlib import Path


# --- IMPORTANT: set this to a known-good R5 jar before importing r5py ---
# Put the jar somewhere local and stable, then point to it here.
# Example name only; adjust to the actual file you have.
R5_JAR = Path("/Users/jedrek/DEDA_LLM_Spatial_Hotelling/data/raw/r5/r5-v7.4-all.jar")

if not R5_JAR.exists():
    raise FileNotFoundError(f"Missing R5 jar: {R5_JAR}")

# JVM / r5py config must be set before import
sys.argv.extend([
    "--r5-classpath", str(R5_JAR),
    "--max-memory", "10G",
    "--verbose",
])
os.environ["JAVA_TOOL_OPTIONS"] = "-Xmx10G -XX:+UseG1GC"

# Clear stale r5py cache before starting JVM
shutil.rmtree(Path.home() / ".cache" / "r5py", ignore_errors=True)

import r5py
'''
from datetime import datetime, timedelta

DEPARTURE = datetime(2025, 10, 7, 10, 0)
WALKING_SPEED_KMH = 4.8

# Origins: compute centroids in the source CRS first, then reproject
origins = grid[["cell_id", "geometry"]].copy()
origins["geometry"] = origins.geometry.centroid
origins = origins.rename(columns={"cell_id": "id"}).to_crs("EPSG:3035")

# Destinations
osm_supermarkets = osm_supermarkets.copy()
osm_supermarkets["store_id"] = osm_supermarkets.index
destinations = (
    osm_supermarkets[["store_id", "geometry"]]
    .rename(columns={"store_id": "id"})
    .to_crs("EPSG:3035")
)

travel_times = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    departure=DEPARTURE,
    departure_time_window=timedelta(hours=2),
    transport_modes=[r5py.TransportMode.TRANSIT, r5py.TransportMode.WALK],
    max_time=timedelta(minutes=60),
    max_time_walking=timedelta(minutes=10),
    speed_walking=WALKING_SPEED_KMH,
    percentiles=[50],
)

In [ ]:
travel_times.to_parquet(PATH_PROCESSED / "travel_times.parquet", index=False)

In [ ]:
travel_times = pd.read_parquet(PATH_PROCESSED / "travel_times.parquet")

In [ ]:
grid_travel_times = grid.merge(
    travel_times[["from_id", "to_id", "travel_time"]],
    left_on="cell_id",
    right_on="from_id",
    how="left"
)
grid_travel_times['if_pop'] = grid_travel_times['Einwohner'] > 0

In [ ]:
osm_supermarkets

In [ ]:
osm_supermarkets
# ID: 491
# (grid_travel_times[grid_travel_times['to_id'] == STORE_ID]['travel_time'].isna())
import numpy as np
STORE_ID = np.random.choice(destinations['id'])
if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=(20,18), dpi=300)
    grid_travel_times[grid_travel_times['to_id'] == STORE_ID].plot(ax=ax, column='travel_time', cmap='viridis', legend=True, alpha =0.5)
    grid_travel_times[(~grid_travel_times['if_pop']) & (grid_travel_times['to_id'] == STORE_ID) & (grid_travel_times[grid_travel_times['to_id'] == STORE_ID]['travel_time'].isna())].plot(ax=ax, color='red', legend=True, alpha =0.5)
    destinations[destinations['id'] == STORE_ID].to_crs(berlin.crs).plot(ax=ax, color='yellow', markersize=5, label=f'Supermarket {STORE_ID}')
    berlin.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=2, label='Berlin Boundary')
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.2)
    ax.set_axis_off()
    ax.legend()
    plt.title(f'Travel Times to Supermarket {STORE_ID}', fontsize=16)
    plt.show()

In [ ]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_05_dist_matrix.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")